<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/01-model-apis/03-streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streaming

**Goal:** Stream responses and understand what UIs need from a streaming backend.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q groq

In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the GROQ_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except ImportError:
    assert os.environ.get('GROQ_API_KEY'), 'Set GROQ_API_KEY'

from groq import Groq
client = Groq()
MODEL = 'openai/gpt-oss-120b'  # update if you get a 404 — see 00-setup/00-environment.ipynb to list available models

## Why streaming

Two latencies matter in an LLM product, and they're wildly different:

- **Time to first token (TTFT)** — how long until the user sees *anything*. Typically well under a second on Groq (which uses custom inference hardware).
- **Total latency** — how long until the response is complete. For a long answer, multiple seconds.

A non-streaming UI makes the user stare at a spinner for the total; a streaming UI feels responsive after the TTFT. Same model, same cost, entirely different product.


In [ ]:
import time

start = time.monotonic()
first_token_at = None

stream = client.chat.completions.create(
    model=MODEL,
    max_tokens=400,
    stream=True,
    messages=[{'role': 'user', 'content': 'Explain, in ~150 words, why database indexes speed up reads but slow down writes.'}],
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        if first_token_at is None:
            first_token_at = time.monotonic() - start
        print(delta, end='', flush=True)

total = time.monotonic() - start
print(f'\n\nTTFT: {first_token_at:.2f}s   total: {total:.2f}s')


Run it and note the gap between the two numbers — that gap is the user experience you buy with streaming. Each `chunk` is a `ChatCompletionChunk`; the text increment lives at `chunk.choices[0].delta.content` (which is `None` on chunks that carry no text, like the first and last).

## The chunk stream anatomy

Groq follows the OpenAI streaming format: a sequence of `ChatCompletionChunk` objects, each carrying a `delta` (the increment since the last chunk):

| Field | Fires | What's in it |
|---|---|---|
| `delta.role` | first chunk | usually `"assistant"`, content empty |
| `delta.content` | many | the text increment (a token or few) |
| `delta.tool_calls` | when calling tools | incremental fragments of the tool call |
| `finish_reason` | last content chunk | `"stop"`, `"length"`, or `"tool_calls"` |
| `x_groq.usage` | final chunk | token accounting (Groq extension) |

Unlike some providers, Groq returns final token usage on the last chunk under `x_groq.usage` — you get incremental display *and* the billing record in one pass, automatically (no extra request flag needed).


In [ ]:
# Same request, inspecting the chunk structure. We accumulate text and read final usage.
text_parts = []
final_usage = None
finish_reason = None

stream = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    stream=True,
    messages=[{'role': 'user', 'content': 'Two sentences: what is connection pooling?'}],
)
for chunk in stream:
    # The usage chunk has an empty choices list, so guard before indexing.
    if chunk.choices:
        delta = chunk.choices[0].delta
        if delta.content:
            text_parts.append(delta.content)
        if chunk.choices[0].finish_reason:
            finish_reason = chunk.choices[0].finish_reason
    # Groq attaches usage to the final chunk under x_groq (sent automatically).
    if getattr(chunk, 'x_groq', None) and chunk.x_groq.usage:
        final_usage = chunk.x_groq.usage

full_text = ''.join(text_parts)
print(full_text)
print()
print('finish_reason:', finish_reason)
print('final usage:', final_usage)


Run it and note two things. First, the usage chunk arrives *after* the content is done and carries an empty `choices` list — that's why we guard `if chunk.choices:` before indexing. Second, you assembled the full text by concatenating deltas yourself; there's no separate "final message" object in the OpenAI streaming format, so accumulating as you go is the pattern.

## Streaming + tool use

Tool calls stream too, and this is where it gets fiddly. The tool call arrives in *fragments* across many chunks: the `id` and function `name` land on the first fragment, then the `arguments` string dribbles in piece by piece over subsequent chunks. You accumulate per `index` and parse when the stream finishes. This is how UIs show "Searching for: berlin weath..." while the model is still writing the call.


In [ ]:
import json

WEATHER_TOOL = {
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get current weather for a city. Call for any weather question.',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
        },
    },
}

# Accumulate tool-call fragments by index. Each index is one tool call.
tool_calls = {}  # index -> {'id': ..., 'name': ..., 'args': ''}

stream = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    stream=True,
    tools=[WEATHER_TOOL],
    messages=[{'role': 'user', 'content': 'What is the weather in Berlin?'}],
)
for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if not delta.tool_calls:
        continue
    for tc in delta.tool_calls:
        slot = tool_calls.setdefault(tc.index, {'id': None, 'name': None, 'args': ''})
        if tc.id:
            slot['id'] = tc.id
        if tc.function and tc.function.name:
            slot['name'] = tc.function.name
            print(f'tool call started: {tc.function.name}')
        if tc.function and tc.function.arguments:
            slot['args'] += tc.function.arguments
            print(f'  partial args so far: {slot["args"]!r}')

# Parse only once the stream is complete — fragments are not valid JSON mid-stream.
for idx, slot in tool_calls.items():
    args = json.loads(slot['args'] or '{}')
    print(f'tool call complete: {slot["name"]}({args})  id={slot["id"]}')


Run it and note the `partial args` fragments — they are *not* valid JSON until the stream ends, so never `json.loads` mid-stream. In practice you accumulate manually only when the UI needs live argument display; otherwise you just collect the full arguments string, parse once, and continue the tool loop exactly as in the previous notebook (execute, append the `role: "tool"` result, open a new stream).

## What a real backend needs

In production you're rarely printing to a terminal — you're sitting between the model and a browser. Three concerns the notebook can't show but you should design for:

**1. Forwarding as SSE.** Don't buffer the whole response server-side; re-emit deltas as your own server-sent events. Sketch (FastAPI-flavored pseudocode — don't run this cell shape in Colab):

```python
@app.post('/chat')
async def chat(req: ChatRequest):
    def gen():
        parts, usage = [], None
        stream = client.chat.completions.create(
            model=MODEL, max_tokens=1000, stream=True, messages=req.messages,
        )
        try:
            for chunk in stream:
                if chunk.choices and chunk.choices[0].delta.content:
                    text = chunk.choices[0].delta.content
                    parts.append(text)
                    yield f'data: {json.dumps({"delta": text})}\n\n'
                if getattr(chunk, 'x_groq', None) and chunk.x_groq.usage:
                    usage = chunk.x_groq.usage
        finally:
            # Runs even if the client disconnected mid-stream:
            record_usage(req.user_id, usage)   # billing truth
        yield 'data: [DONE]\n\n'
    return StreamingResponse(gen(), media_type='text/event-stream')
```

**2. Client disconnects.** Users close tabs mid-answer constantly. Your generator gets cancelled — but the model kept generating and you keep paying. Decide deliberately: abort the upstream request on disconnect (saves tokens, loses the answer) or let it finish and persist the result (costs tokens, enables "resume"). Either way, the `finally` block must still capture usage.

**3. Usage for billing.** The usage chunk arrives *last* — if you only forward text deltas and drop the tail chunk, you've thrown away the bill. Always read the `x_groq.usage` chunk server-side and record it before the request handler exits.


## Exercises

1. Extend the TTFT cell to also record *inter-token gaps* (time between consecutive deltas) and print p50/p95. This is the number that makes streamed output feel smooth or janky.
2. Build `stream_to_list(prompt)` that returns `(chunks, final_usage)` — every text delta in order plus the final usage object. Verify the joined chunks equal the full assembled text.
3. Combine this notebook with the previous one: a `run_agent_streaming` loop that streams each turn, prints text deltas live, then reassembles the tool calls from the fragments and continues until `finish_reason == 'stop'`.
4. Simulate a client disconnect: break out of the chunk loop after the 10th text delta. Confirm you lose the usage chunk (it comes last) — then restructure so a `finally` block still records what you streamed so far.
